In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

# Load and prep active users

df2 = pd.read_excel(
    "../mock-data/Active_Inactive_Non-Diala_Mock.xlsx",
    sheet_name="Active_Users"
)
df2 = df2.dropna(how="all")
print(f"Active raw shape: {df2.shape}")

# Numeric coerce
df2["Diab_Duration"] = pd.to_numeric(df2["Diab_Duration"], errors="coerce")
for col in ["AGE", "AOS", "Diab_Duration",
            "HbA1c_BL", "HbA1c_FU",
            "BMI_BL", "BMI_FU",
            "HDL_BL", "HDL_FU",
            "TGL_BL", "TGL_FU",
            "Serum_Cholesterol_BL", "Serum_Cholesterol_FU",
            "Kupp_Occupation"]:
    if col in df2.columns:
        df2[col] = pd.to_numeric(df2[col], errors="coerce")

# Replace 0 with NaN for lab cols
zero_cols = ["TGL_BL", "TGL_FU", "Serum_Cholesterol_BL", "Serum_Cholesterol_FU",
             "HDL_BL", "HDL_FU"]
for col in zero_cols:
    if col in df2.columns:
        df2[col] = df2[col].replace(0, np.nan)

# String clean
replace_vals = ["", " ", "NA", "N/A", "na", "n/a", "NIL", "nil",
                "None", "none", "NULL", "null", "-", "--", "nan"]
for col in df2.select_dtypes(include="object").columns:
    df2[col] = df2[col].str.strip()
    df2[col] = df2[col].replace(replace_vals, np.nan)

# Kuppuswamy score 
kupp_map = {
    "Politician": 10, "Manager": 10, "Central Government Service": 10,
    "State Government Service": 10, "Tahsildar": 10,
    "Doctor": 9, "Doctor-Dentist": 9, "Doctor-General Physician": 9,
    "Doctor-Ophthalmologist": 9, "Doctor-Homeopathist": 9,
    "Doctor-Pediatrician": 9, "Doctor-Gynecologist": 9,
    "Doctor-Surgeon": 9, "Doctor-Neurologist": 9,
    "Doctor-Siddha": 9, "Doctor-Ayurvedic": 9,
    "Advocate": 9, "Architecture": 9, "AUDITOR": 9,
    "Engineer": 9, "IT Professional": 9,
    "Professor / Teacher / Education": 9, "Doctorate": 9,
    "Reporter": 8, "Accounts/Finance": 8, "Bank": 8,
    "IT Employee": 8, "Supervisor": 8, "Armed Forces": 8, "Police": 8,
    "Clerk": 7, "Government": 7,
    "Business": 6, "Self Employed": 6, "Fashion / Saloon": 6,
    "Retired Employee": 6, "Father In Church": 6, "Priest": 6, "Social Service": 6,
    "Farmer / Agriculture": 5,
    "Private Sector": 4,
    "Driver": 3, "Courier": 3,
    "Daily wages": 2,
    "Housewife": 1, "Retired": 1, "Armed Forces-Retired": 1, "Student": 1,
}
df2["Kupp_Occupation"] = df2["Occupation"].map(kupp_map)
df2 = df2.dropna(subset=["Kupp_Occupation"])

# Recode
df2["Gender"]      = df2["Gender"].map({"M": "Male", "F": "Female"})
df2["Gender_Code"] = df2["Gender"].map({"Male": 1, "Female": 2})

def age_group(age):
    if pd.isna(age):  return np.nan
    elif age < 30:    return 0
    elif age < 40:    return 1
    elif age < 50:    return 2
    elif age < 60:    return 3
    else:             return 4

age_labels = {0: "<30", 1: "30–39", 2: "40–49", 3: "50–59", 4: "≥60"}
df2["Age_Group"]     = df2["AGE"].apply(age_group)
df2["Age_Group_str"] = df2["Age_Group"].map(age_labels)
df2["SES_Group"]     = pd.cut(
    df2["Kupp_Occupation"], bins=[0, 4, 7, 10],
    labels=["Low (1–4)", "Middle (5–7)", "High (8–10)"]
)
df2["SES_Group_str"]  = df2["SES_Group"].astype(str)
df2["Delta_HbA1c"]    = df2["HbA1c_FU"]            - df2["HbA1c_BL"]
df2["Delta_BMI"]      = df2["BMI_FU"]               - df2["BMI_BL"]
df2["Delta_HDL"]      = df2["HDL_FU"]               - df2["HDL_BL"]
df2["Delta_TGL"]      = df2["TGL_FU"]               - df2["TGL_BL"]
df2["Delta_Serum_Cholesterol"] = df2["Serum_Cholesterol_FU"] - df2["Serum_Cholesterol_BL"]

df2 = df2[df2["HbA1c_BL"].notna() & df2["HbA1c_FU"].notna()].copy()
print(f"Active final n: {len(df2)}")

# Load and prep non-diala users

df_nd = pd.read_excel(
    "../mock-data/Active_Inactive_Non-Diala_Mock.xlsx",
    sheet_name="Non_DiaLA_Users"
)
df_nd = df_nd.dropna(how="all")
print(f"\nNon-DiaLA raw shape: {df_nd.shape}")

# Rename to standardised names
df_nd = df_nd.rename(columns={
    "GENDER":                     "Gender",
    "FU_DATE":                    "FU_Entry",
    "HDL CHOLESTEROL_BL":         "HDL_BL",
    "SERUM TRIGLYCERIDES_BL":     "TGL_BL",
    "HDL CHOLESTEROL_FU":         "HDL_FU",
    "SERUM TRIGLYCERIDES_FU":     "TGL_FU",
    "Serum Cholesterol_BL":       "Serum_Cholesterol_BL",
    "Serum Cholesterol_FU":       "Serum_Cholesterol_FU",
})

# Numeric coerce
for col in ["AGE",
            "HbA1c_BL", "HbA1c_FU",
            "BMI_BL", "BMI_FU",
            "HDL_BL", "HDL_FU",
            "TGL_BL", "TGL_FU",
            "Serum_Cholesterol_BL", "Serum_Cholesterol_FU"]:
    if col in df_nd.columns:
        df_nd[col] = pd.to_numeric(df_nd[col], errors="coerce")

# Replace 0 with NaN for lab cols
for col in zero_cols:
    if col in df_nd.columns:
        df_nd[col] = df_nd[col].replace(0, np.nan)

# String clean
for col in df_nd.select_dtypes(include="object").columns:
    df_nd[col] = df_nd[col].str.strip()
    df_nd[col] = df_nd[col].replace(replace_vals, np.nan)

# Kuppuswamy
df_nd["Kupp_Occupation"] = df_nd["Occupation"].map(kupp_map)
df_nd = df_nd.dropna(subset=["Kupp_Occupation"])

# Recode
df_nd["Gender"]      = df_nd["Gender"].map({"M": "Male", "F": "Female"})
df_nd["Gender_Code"] = df_nd["Gender"].map({"Male": 1, "Female": 2})
df_nd["Age_Group"]     = df_nd["AGE"].apply(age_group)
df_nd["Age_Group_str"] = df_nd["Age_Group"].map(age_labels)
df_nd["SES_Group"]     = pd.cut(
    df_nd["Kupp_Occupation"], bins=[0, 4, 7, 10],
    labels=["Low (1–4)", "Middle (5–7)", "High (8–10)"]
)
df_nd["SES_Group_str"]  = df_nd["SES_Group"].astype(str)
df_nd["Delta_HbA1c"]    = df_nd["HbA1c_FU"]            - df_nd["HbA1c_BL"]
df_nd["Delta_BMI"]      = df_nd["BMI_FU"]               - df_nd["BMI_BL"]
df_nd["Delta_HDL"]      = df_nd["HDL_FU"]               - df_nd["HDL_BL"]
df_nd["Delta_TGL"]      = df_nd["TGL_FU"]               - df_nd["TGL_BL"]
df_nd["Delta_Serum_Cholesterol"] = df_nd["Serum_Cholesterol_FU"] - df_nd["Serum_Cholesterol_BL"]

df_nd = df_nd[df_nd["HbA1c_BL"].notna() & df_nd["HbA1c_FU"].notna()].copy()
print(f"Non-DiaLA final n: {len(df_nd)}")

# Like I did previously, I combine the two datasets into a single dataframe for comparison.

shared_cols = [
    "MRNO", "Gender", "Gender_Code",
    "AGE", "Age_Group", "Age_Group_str",
    "Kupp_Occupation", "SES_Group", "SES_Group_str",
    # HbA1c
    "HbA1c_BL", "HbA1c_FU", "Delta_HbA1c",
    # BMI
    "BMI_BL", "BMI_FU", "Delta_BMI",
    # HDL
    "HDL_BL", "HDL_FU", "Delta_HDL",
    # TGL
    "TGL_BL", "TGL_FU", "Delta_TGL",
    # Serum Cholesterol
    "Serum_Cholesterol_BL", "Serum_Cholesterol_FU", "Delta_Serum_Cholesterol",
]

df_active   = df2[[c for c in shared_cols if c in df2.columns]].copy()
df_nondiala = df_nd[[c for c in shared_cols if c in df_nd.columns]].copy()

df_active["Group"]   = "Active"
df_nondiala["Group"] = "Non-DiaLA"

df_combined = pd.concat([df_active, df_nondiala], ignore_index=True)

print(f"\nCombined shape: {df_combined.shape}")
print(f"Active n:    {(df_combined['Group'] == 'Active').sum()}")
print(f"Non-DiaLA n: {(df_combined['Group'] == 'Non-DiaLA').sum()}")
print(f"\nColumns: {list(df_combined.columns)}")
print(df_combined[["MRNO", "Group", "Gender", "AGE", "Kupp_Occupation",
                    "HbA1c_BL", "HbA1c_FU", "Delta_HbA1c",
                    "BMI_BL",   "BMI_FU",   "Delta_BMI"]].head(10).to_string())

In [ ]:
# MODULE 1: DESCRIPTIVE COMPARISON — ACTIVE vs NON-DIALA

print("=" * 70)
print("MODULE 1: DESCRIPTIVE COMPARISON — ACTIVE vs NON-DIALA")
print("=" * 70)

active   = df_combined[df_combined["Group"] == "Active"]
nondiala = df_combined[df_combined["Group"] == "Non-DiaLA"]

# Continuous variables 
continuous = {
    "Age (years)":          "AGE",
    "Kuppuswamy SES Score": "Kupp_Occupation",
    # HbA1c
    "HbA1c BL (%)":        "HbA1c_BL",
    "HbA1c FU (%)":        "HbA1c_FU",
    "Delta HbA1c":          "Delta_HbA1c",
    # BMI
    "BMI BL (kg/m²)":      "BMI_BL",
    "BMI FU (kg/m²)":      "BMI_FU",
    "Delta BMI":            "Delta_BMI",
    # HDL
    "HDL BL (mg/dL)":      "HDL_BL",
    "HDL FU (mg/dL)":      "HDL_FU",
    "Delta HDL":            "Delta_HDL",
    # TGL
    "TGL BL (mg/dL)":      "TGL_BL",
    "TGL FU (mg/dL)":      "TGL_FU",
    "Delta TGL":            "Delta_TGL",
    # Serum Cholesterol
    "Serum Chol BL":        "Serum_Cholesterol_BL",
    "Serum Chol FU":        "Serum_Cholesterol_FU",
    "Delta Serum Chol":     "Delta_Serum_Cholesterol",
}

print(f"\n{'Variable':<30} {'Active (Mean±SD)':>22} {'Non-DiaLA (Mean±SD)':>22} {'Active n':>10} {'NonDiaLA n':>10}")
print("-" * 96)
for label, col in continuous.items():
    if col in df_combined.columns:
        a  = active[col].dropna()
        nd = nondiala[col].dropna()
        a_str  = f"{a.mean():.2f} ± {a.std():.2f}" if len(a)  > 0 else "N/A"
        nd_str = f"{nd.mean():.2f} ± {nd.std():.2f}" if len(nd) > 0 else "N/A"
        print(f"{label:<30} {a_str:>22} {nd_str:>22} {len(a):>10} {len(nd):>10}")
    else:
        print(f"{label:<30} {'— not in dataset —':>22} {'':>22}")

# Categorical variables 
print("\n" + "=" * 70)
print("CATEGORICAL COMPARISON")
print("=" * 70)

# Gender
print("\nGender:")
print(f"  {'':20} {'Active':>14} {'Non-DiaLA':>14}")
print(f"  {'-'*50}")
for val in ["Male", "Female"]:
    a_n   = (active["Gender"]   == val).sum()
    nd_n  = (nondiala["Gender"] == val).sum()
    a_pct  = a_n  / len(active)   * 100
    nd_pct = nd_n / len(nondiala) * 100
    print(f"  {val:<20} {a_n:>5} ({a_pct:.1f}%)  {nd_n:>5} ({nd_pct:.1f}%)")

# Age Group
print("\nAge Group:")
print(f"  {'':20} {'Active':>14} {'Non-DiaLA':>14}")
print(f"  {'-'*50}")
for val in sorted(df_combined["Age_Group"].dropna().unique()):
    label = age_labels[val]
    a_n   = (active["Age_Group"]   == val).sum()
    nd_n  = (nondiala["Age_Group"] == val).sum()
    a_pct  = a_n  / len(active)   * 100
    nd_pct = nd_n / len(nondiala) * 100
    print(f"  {label:<20} {a_n:>5} ({a_pct:.1f}%)  {nd_n:>5} ({nd_pct:.1f}%)")

# SES Group
print("\nSES Group:")
print(f"  {'':20} {'Active':>14} {'Non-DiaLA':>14}")
print(f"  {'-'*50}")
ses_order = ["Low (1–4)", "Middle (5–7)", "High (8–10)"]
for val in ses_order:
    a_n   = (active["SES_Group_str"]   == val).sum()
    nd_n  = (nondiala["SES_Group_str"] == val).sum()
    a_pct  = a_n  / len(active)   * 100
    nd_pct = nd_n / len(nondiala) * 100
    print(f"  {val:<20} {a_n:>5} ({a_pct:.1f}%)  {nd_n:>5} ({nd_pct:.1f}%)")

In [ ]:
# MODULE 2: BASELINE COMPARISON — ACTIVE vs NON-DIALA (ALL OUTCOMES)

print("=" * 70)
print("MODULE 2: BASELINE COMPARISON — ACTIVE vs NON-DIALA")
print("=" * 70)

active   = df_combined[df_combined["Group"] == "Active"]
nondiala = df_combined[df_combined["Group"] == "Non-DiaLA"]

baseline_outcomes = {
    "HbA1c":            ("HbA1c_BL",            "HbA1c (%)"),
    "BMI":              ("BMI_BL",               "BMI (kg/m²)"),
    "HDL":              ("HDL_BL",               "HDL (mg/dL)"),
    "TGL":              ("TGL_BL",               "TGL (mg/dL)"),
    "Serum_Cholesterol":("Serum_Cholesterol_BL", "Serum Cholesterol (mg/dL)"),
}

module2_results = []

for name, (col, units) in baseline_outcomes.items():

    a_bl  = active[col].dropna()
    nd_bl = nondiala[col].dropna()

    if len(a_bl) < 3 or len(nd_bl) < 3:
        print(f"\n{name}: insufficient data, skipping.")
        continue

    print(f"\n{'─' * 70}")
    print(f"{name} ({units})")
    print(f"  Active     n={len(a_bl):>6}  mean={a_bl.mean():.2f}  "
          f"sd={a_bl.std():.2f}  median={a_bl.median():.2f}")
    print(f"  Non-DiaLA  n={len(nd_bl):>6}  mean={nd_bl.mean():.2f}  "
          f"sd={nd_bl.std():.2f}  median={nd_bl.median():.2f}")

    # Normality 
    _, p_a  = stats.shapiro(a_bl.sample(min(len(a_bl),  5000), random_state=42))
    _, p_nd = stats.shapiro(nd_bl.sample(min(len(nd_bl), 5000), random_state=42))
    print(f"\n  Shapiro Active:    p={p_a:.4f}  ({'normal' if p_a > 0.05 else 'non-normal'})")
    print(f"  Shapiro Non-DiaLA: p={p_nd:.4f}  ({'normal' if p_nd > 0.05 else 'non-normal'})")

    # Mann-Whitney U 
    u_stat, u_p = stats.mannwhitneyu(a_bl, nd_bl, alternative="two-sided")
    u_sig = "***" if u_p < 0.001 else "**" if u_p < 0.01 else "*" if u_p < 0.05 else "ns"
    print(f"\n  Mann-Whitney U={u_stat:.1f}  p={u_p:.4f}  {u_sig}")

    if u_p < 0.05:
        direction = "Active higher" if a_bl.mean() > nd_bl.mean() else "Non-DiaLA higher"
        print(f"  ► Groups differ significantly at baseline ({direction})")
        print(f"    Account for this when interpreting outcome differences in Module 3")
    else:
        print(f"  ► Groups comparable at baseline")

    module2_results.append({
        "Variable":       name,
        "Active_n":       len(a_bl),
        "Active_mean":    round(a_bl.mean(), 2),
        "Active_sd":      round(a_bl.std(), 2),
        "NonDiaLA_n":     len(nd_bl),
        "NonDiaLA_mean":  round(nd_bl.mean(), 2),
        "NonDiaLA_sd":    round(nd_bl.std(), 2),
        "U_stat":         round(u_stat, 1),
        "p_value":        round(u_p, 4),
        "Sig":            u_sig,
    })

# Summary table 
print(f"\n{'=' * 70}")
print("MODULE 2 SUMMARY TABLE — BASELINE COMPARISONS")
print(f"{'=' * 70}")
print(f"{'Variable':<22} {'Active Mean±SD':>18} {'NonDiaLA Mean±SD':>18} "
      f"{'U stat':>10} {'p':>8} {'Sig':>5}")
print("-" * 85)
for r in module2_results:
    a_str  = f"{r['Active_mean']:.2f} ± {r['Active_sd']:.2f}"
    nd_str = f"{r['NonDiaLA_mean']:.2f} ± {r['NonDiaLA_sd']:.2f}"
    print(f"{r['Variable']:<22} {a_str:>18} {nd_str:>18} "
          f"{r['U_stat']:>10} {r['p_value']:>8} {r['Sig']:>5}")

# Box plots (one per outcome) 
n_outcomes = len(baseline_outcomes)
fig, axes  = plt.subplots(1, n_outcomes, figsize=(5 * n_outcomes, 5))
colors     = ["#5cb85c", "#2E86AB"]   # green = Active, blue = Non-DiaLA

for ax, (name, (col, units)) in zip(axes, baseline_outcomes.items()):
    a_bl  = active[col].dropna().values
    nd_bl = nondiala[col].dropna().values

    bp = ax.boxplot([a_bl, nd_bl], tick_labels=["Active", "Non-DiaLA"],
                    patch_artist=True)
    for patch, color in zip(bp["boxes"], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)

    # significance bar
    r = next(r for r in module2_results if r["Variable"] == name)
    y_max = max(a_bl.max(), nd_bl.max()) * 1.05
    ax.plot([1, 2], [y_max, y_max], color="black", linewidth=1)
    ax.text(1.5, y_max * 1.01, r["Sig"], ha="center",
            fontsize=12, fontweight="bold")

    ax.set_title(f"{name}\n{units}", fontsize=10, fontweight="bold")
    ax.set_ylabel(units, fontsize=9)
    ax.tick_params(axis="x", rotation=15)

plt.suptitle("Baseline Clinical Values — Active vs Non-DiaLA",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("comparison_module2_baseline_all.png", dpi=150, bbox_inches="tight")
plt.close()
print("\n✔ Plot saved — open comparison_module2_baseline_all.png")
print("\n✔ Module 2 complete — paste output and we move to Module 3.")

In [ ]:
# MODULE 3: DELTA COMPARISON — ACTIVE vs NON-DIALA (ALL OUTCOMES)

print("=" * 70)
print("MODULE 3: DELTA COMPARISON — ACTIVE vs NON-DIALA")
print("=" * 70)
print("""
NOTE: Check Module 2 — if groups differed at baseline, interpret
      delta differences cautiously (regression to mean may apply).
""")

active   = df_combined[df_combined["Group"] == "Active"]
nondiala = df_combined[df_combined["Group"] == "Non-DiaLA"]

delta_outcomes = {
    "HbA1c":             ("Delta_HbA1c",            "HbA1c_BL",            "HbA1c_FU",            "negative", "HbA1c (%)"),
    "BMI":               ("Delta_BMI",               "BMI_BL",              "BMI_FU",              "negative", "BMI (kg/m²)"),
    "HDL":               ("Delta_HDL",               "HDL_BL",              "HDL_FU",              "positive", "HDL (mg/dL)"),
    "TGL":               ("Delta_TGL",               "TGL_BL",              "TGL_FU",              "negative", "TGL (mg/dL)"),
    "Serum_Cholesterol": ("Delta_Serum_Cholesterol",  "Serum_Cholesterol_BL","Serum_Cholesterol_FU","negative", "Serum Chol (mg/dL)"),
}

module3_results = []

for name, (delta_col, bl_col, fu_col, direction, units) in delta_outcomes.items():

    a_delta  = active[delta_col].dropna()
    nd_delta = nondiala[delta_col].dropna()

    if len(a_delta) < 3 or len(nd_delta) < 3:
        print(f"\n{name}: insufficient data, skipping.")
        continue

    print(f"\n{'─' * 70}")
    print(f"{name} ({units})")
    print(f"  Active     n={len(a_delta):>6}  mean={a_delta.mean():.3f}  "
          f"sd={a_delta.std():.3f}  median={a_delta.median():.3f}")
    print(f"  Non-DiaLA  n={len(nd_delta):>6}  mean={nd_delta.mean():.3f}  "
          f"sd={nd_delta.std():.3f}  median={nd_delta.median():.3f}")

    # Direction 
    def improved(mean, direction):
        if direction == "negative": return "✓ improved (↓)" if mean < 0 else "✗ worsened (↑)"
        else:                       return "✓ improved (↑)" if mean > 0 else "✗ worsened (↓)"

    print(f"\n  Active     delta: {improved(a_delta.mean(),  direction)}")
    print(f"  Non-DiaLA  delta: {improved(nd_delta.mean(), direction)}")

    # Normality 
    _, p_a  = stats.shapiro(a_delta.sample(min(len(a_delta),  5000), random_state=42))
    _, p_nd = stats.shapiro(nd_delta.sample(min(len(nd_delta), 5000), random_state=42))
    print(f"\n  Shapiro Active:    p={p_a:.4f}  ({'normal' if p_a > 0.05 else 'non-normal'})")
    print(f"  Shapiro Non-DiaLA: p={p_nd:.4f}  ({'normal' if p_nd > 0.05 else 'non-normal'})")

    # Mann-Whitney U 
    u_stat, u_p = stats.mannwhitneyu(a_delta, nd_delta, alternative="two-sided")
    u_sig = "***" if u_p < 0.001 else "**" if u_p < 0.01 else "*" if u_p < 0.05 else "ns"

    # Effect size 
    rbc  = 1 - (2 * u_stat) / (len(a_delta) * len(nd_delta))
    mag  = "small" if abs(rbc) < 0.3 else "medium" if abs(rbc) < 0.5 else "large"

    print(f"\n  Mann-Whitney U={u_stat:.1f}  p={u_p:.4f}  {u_sig}")
    print(f"  Rank biserial r={rbc:.3f}  ({mag} effect)")

    # Baseline means for context 
    a_bl_mean  = active[bl_col].mean()
    nd_bl_mean = nondiala[bl_col].mean()
    print(f"\n  Baseline context — Active BL: {a_bl_mean:.2f}  |  Non-DiaLA BL: {nd_bl_mean:.2f}")

    # Conclusion 
    print(f"\n  ► ", end="")
    if u_p < 0.05:
        if direction == "negative":
            better = "Active" if a_delta.mean() < nd_delta.mean() else "Non-DiaLA"
        else:
            better = "Active" if a_delta.mean() > nd_delta.mean() else "Non-DiaLA"
        print(f"{better} showed significantly greater {name} improvement")
    else:
        print(f"No significant difference in {name} improvement between groups")

    module3_results.append({
        "Variable":       name,
        "Active_n":       len(a_delta),
        "Active_delta":   round(a_delta.mean(), 3),
        "NonDiaLA_n":     len(nd_delta),
        "NonDiaLA_delta": round(nd_delta.mean(), 3),
        "U_stat":         round(u_stat, 1),
        "p_value":        round(u_p, 4),
        "Sig":            u_sig,
        "Effect_r":       round(rbc, 3),
        "Effect_mag":     mag,
        "direction":      direction,
        "bl_col":         bl_col,
        "fu_col":         fu_col,
        "delta_col":      delta_col,
        "units":          units,
    })

# Summary table 
print(f"\n{'=' * 70}")
print("MODULE 3 SUMMARY TABLE — DELTA COMPARISONS")
print(f"{'=' * 70}")
print(f"{'Variable':<22} {'Active Δ':>10} {'NonDiaLA Δ':>12} "
      f"{'U stat':>10} {'p':>8} {'Sig':>5} {'Effect r':>10} {'Magnitude'}")
print("-" * 90)
for r in module3_results:
    print(f"{r['Variable']:<22} {r['Active_delta']:>10} {r['NonDiaLA_delta']:>12} "
          f"{r['U_stat']:>10} {r['p_value']:>8} {r['Sig']:>5} "
          f"{r['Effect_r']:>10} {r['Effect_mag']}")

# PLOTS: 2-panel per outcome (delta boxplot + BL→FU trajectory)

colors = ["#5cb85c", "#2E86AB"]   # green = Active, blue = Non-DiaLA
n      = len(module3_results)

fig, axes = plt.subplots(n, 2, figsize=(12, 5 * n))

for row, r in enumerate(module3_results):
    name      = r["Variable"]
    delta_col = r["delta_col"]
    bl_col    = r["bl_col"]
    fu_col    = r["fu_col"]
    units     = r["units"]
    u_sig     = r["Sig"]

    a_delta  = active[delta_col].dropna().values
    nd_delta = nondiala[delta_col].dropna().values

    # Panel 1: Delta boxplot 
    ax = axes[row, 0]
    bp = ax.boxplot([a_delta, nd_delta],
                    tick_labels=["Active", "Non-DiaLA"],
                    patch_artist=True)
    for patch, color in zip(bp["boxes"], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax.axhline(0, color="black", linestyle="--", linewidth=0.8, alpha=0.5)
    y_max = max(np.percentile(a_delta, 95), np.percentile(nd_delta, 95)) * 1.1
    ax.set_ylim(top=y_max * 1.15)
    ax.plot([1, 2], [y_max, y_max], color="black", linewidth=1)
    ax.text(1.5, y_max * 1.02, u_sig, ha="center", fontsize=12, fontweight="bold")
    ax.set_title(f"{name} — Delta Comparison", fontweight="bold")
    ax.set_ylabel(f"Δ {units}")
    ax.set_xlabel("Group")

    # Panel 2: BL → FU trajectory 
    ax = axes[row, 1]
    x = np.array([0, 1])
    a_means  = [active[bl_col].mean(),   active[fu_col].mean()]
    nd_means = [nondiala[bl_col].mean(), nondiala[fu_col].mean()]
    a_sds    = [active[bl_col].std(),    active[fu_col].std()]
    nd_sds   = [nondiala[bl_col].std(),  nondiala[fu_col].std()]

    ax.errorbar(x - 0.05, a_means,  yerr=a_sds,  fmt="-o", color=colors[0],
                label="Active",    linewidth=2, capsize=5, markersize=8)
    ax.errorbar(x + 0.05, nd_means, yerr=nd_sds, fmt="-o", color=colors[1],
                label="Non-DiaLA", linewidth=2, capsize=5, markersize=8)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(["Baseline", "Follow-up"])
    ax.set_title(f"{name} — BL → FU Trajectory", fontweight="bold")
    ax.set_ylabel(units)
    ax.legend()

plt.suptitle("Clinical Improvement — Active vs Non-DiaLA",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("comparison_module3_delta_all.png", dpi=150, bbox_inches="tight")
plt.close()
print("\n✔ Plot saved — open comparison_module3_delta_all.png")
print("\n✔ Module 3 complete — paste output and we move to Module 4.")

In [ ]:
# MODULE 4: FOLLOW-UP COMPARISON — ACTIVE vs NON-DIALA (ALL OUTCOMES)

print("=" * 70)
print("MODULE 4: FOLLOW-UP COMPARISON — ACTIVE vs NON-DIALA")
print("=" * 70)

active   = df_combined[df_combined["Group"] == "Active"]
nondiala = df_combined[df_combined["Group"] == "Non-DiaLA"]

fu_outcomes = {
    "HbA1c":             ("HbA1c_FU",            "HbA1c_BL",            "Delta_HbA1c",            "negative", "HbA1c (%)"),
    "BMI":               ("BMI_FU",               "BMI_BL",              "Delta_BMI",               "negative", "BMI (kg/m²)"),
    "HDL":               ("HDL_FU",               "HDL_BL",              "Delta_HDL",               "positive", "HDL (mg/dL)"),
    "TGL":               ("TGL_FU",               "TGL_BL",              "Delta_TGL",               "negative", "TGL (mg/dL)"),
    "Serum_Cholesterol": ("Serum_Cholesterol_FU",  "Serum_Cholesterol_BL","Delta_Serum_Cholesterol", "negative", "Serum Chol (mg/dL)"),
}

module4_results = []
colors = ["#5cb85c", "#2E86AB"]   # green = Active, blue = Non-DiaLA

for name, (fu_col, bl_col, delta_col, direction, units) in fu_outcomes.items():

    a_fu  = active[fu_col].dropna()
    nd_fu = nondiala[fu_col].dropna()

    if len(a_fu) < 3 or len(nd_fu) < 3:
        print(f"\n{name}: insufficient data, skipping.")
        continue

    print(f"\n{'─' * 70}")
    print(f"{name} ({units})")
    print(f"  Active     n={len(a_fu):>6}  mean={a_fu.mean():.2f}  "
          f"sd={a_fu.std():.2f}  median={a_fu.median():.2f}")
    print(f"  Non-DiaLA  n={len(nd_fu):>6}  mean={nd_fu.mean():.2f}  "
          f"sd={nd_fu.std():.2f}  median={nd_fu.median():.2f}")

    # Normality 
    _, p_a  = stats.shapiro(a_fu.sample(min(len(a_fu),  5000), random_state=42))
    _, p_nd = stats.shapiro(nd_fu.sample(min(len(nd_fu), 5000), random_state=42))
    print(f"\n  Shapiro Active:    p={p_a:.4f}  ({'normal' if p_a > 0.05 else 'non-normal'})")
    print(f"  Shapiro Non-DiaLA: p={p_nd:.4f}  ({'normal' if p_nd > 0.05 else 'non-normal'})")

    # Mann-Whitney U 
    u_stat, u_p = stats.mannwhitneyu(a_fu, nd_fu, alternative="two-sided")
    u_sig = "***" if u_p < 0.001 else "**" if u_p < 0.01 else "*" if u_p < 0.05 else "ns"
    rbc   = 1 - (2 * u_stat) / (len(a_fu) * len(nd_fu))
    mag   = "small" if abs(rbc) < 0.3 else "medium" if abs(rbc) < 0.5 else "large"

    print(f"\n  Mann-Whitney U={u_stat:.1f}  p={u_p:.4f}  {u_sig}")
    print(f"  Rank biserial r={rbc:.3f}  ({mag} effect)")

    # Context from Modules 2 & 3 
    print(f"\n  ► Context (Modules 2 & 3):")
    print(f"    Baseline — Active: {active[bl_col].mean():.2f}  |  "
          f"Non-DiaLA: {nondiala[bl_col].mean():.2f}")
    print(f"    Delta    — Active: {active[delta_col].mean():.3f}  |  "
          f"Non-DiaLA: {nondiala[delta_col].mean():.3f}")
    print(f"    Follow-up— Active: {a_fu.mean():.2f}  |  "
          f"Non-DiaLA: {nd_fu.mean():.2f}")

    # Conclusion 
    print(f"\n  ► ", end="")
    if u_p < 0.05:
        if direction == "negative":
            better = "Active" if a_fu.mean() < nd_fu.mean() else "Non-DiaLA"
        else:
            better = "Active" if a_fu.mean() > nd_fu.mean() else "Non-DiaLA"
        print(f"{better} had significantly better {name} at follow-up")
    else:
        print(f"No significant difference in follow-up {name} between groups")

    module4_results.append({
        "Variable":      name,
        "Active_n":      len(a_fu),
        "Active_FU":     round(a_fu.mean(), 2),
        "Active_BL":     round(active[bl_col].mean(), 2),
        "Active_delta":  round(active[delta_col].mean(), 3),
        "NonDiaLA_n":    len(nd_fu),
        "NonDiaLA_FU":   round(nd_fu.mean(), 2),
        "NonDiaLA_BL":   round(nondiala[bl_col].mean(), 2),
        "NonDiaLA_delta":round(nondiala[delta_col].mean(), 3),
        "U_stat":        round(u_stat, 1),
        "p_value":       round(u_p, 4),
        "Sig":           u_sig,
        "Effect_r":      round(rbc, 3),
        "Effect_mag":    mag,
        "bl_col":        bl_col,
        "fu_col":        fu_col,
        "delta_col":     delta_col,
        "units":         units,
    })

# Summary table 
print(f"\n{'=' * 70}")
print("MODULE 4 SUMMARY TABLE — FOLLOW-UP COMPARISONS")
print(f"{'=' * 70}")
print(f"{'Variable':<22} {'Active FU':>10} {'NonDiaLA FU':>12} "
      f"{'U stat':>10} {'p':>8} {'Sig':>5} {'Effect r':>10} {'Magnitude'}")
print("-" * 90)
for r in module4_results:
    print(f"{r['Variable']:<22} {r['Active_FU']:>10} {r['NonDiaLA_FU']:>12} "
          f"{r['U_stat']:>10} {r['p_value']:>8} {r['Sig']:>5} "
          f"{r['Effect_r']:>10} {r['Effect_mag']}")

# PLOTS: 2-panel per outcome (FU boxplot + BL→FU trajectory with delta)

n   = len(module4_results)
fig, axes = plt.subplots(n, 2, figsize=(12, 5 * n))

for row, r in enumerate(module4_results):
    name      = r["Variable"]
    fu_col    = r["fu_col"]
    bl_col    = r["bl_col"]
    delta_col = r["delta_col"]
    units     = r["units"]
    u_sig     = r["Sig"]

    a_fu  = active[fu_col].dropna().values
    nd_fu = nondiala[fu_col].dropna().values

    # Panel 1: FU boxplot 
    ax = axes[row, 0]
    bp = ax.boxplot([a_fu, nd_fu],
                    tick_labels=["Active", "Non-DiaLA"],
                    patch_artist=True)
    for patch, color in zip(bp["boxes"], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    y_max = max(np.percentile(a_fu, 95), np.percentile(nd_fu, 95)) * 1.1
    ax.set_ylim(top=y_max * 1.15)
    ax.plot([1, 2], [y_max, y_max], color="black", linewidth=1)
    ax.text(1.5, y_max * 1.02, u_sig, ha="center", fontsize=12, fontweight="bold")
    ax.set_title(f"{name} — Follow-up Comparison", fontweight="bold")
    ax.set_ylabel(units)
    ax.set_xlabel("Group")

    # Panel 2: BL → FU trajectory with delta annotations 
    ax = axes[row, 1]
    x = np.array([0, 1])

    a_bl_m  = active[bl_col].mean();    a_fu_m  = active[fu_col].mean()
    nd_bl_m = nondiala[bl_col].mean();  nd_fu_m = nondiala[fu_col].mean()
    a_bl_s  = active[bl_col].std();     a_fu_s  = active[fu_col].std()
    nd_bl_s = nondiala[bl_col].std();   nd_fu_s = nondiala[fu_col].std()

    ax.errorbar(x - 0.05, [a_bl_m, a_fu_m],   yerr=[a_bl_s, a_fu_s],
                fmt="-o", color=colors[0], label="Active",
                linewidth=2, capsize=5, markersize=8)
    ax.errorbar(x + 0.05, [nd_bl_m, nd_fu_m], yerr=[nd_bl_s, nd_fu_s],
                fmt="-o", color=colors[1], label="Non-DiaLA",
                linewidth=2, capsize=5, markersize=8)

    # Delta annotations
    ax.annotate(f"Δ={active[delta_col].mean():.2f}",
                xy=(1 - 0.05, a_fu_m),
                xytext=(0.55, a_fu_m + (a_fu_s * 0.3)),
                fontsize=9, color=colors[0],
                arrowprops=dict(arrowstyle="->", color=colors[0]))
    ax.annotate(f"Δ={nondiala[delta_col].mean():.2f}",
                xy=(1 + 0.05, nd_fu_m),
                xytext=(1.1, nd_fu_m + (nd_fu_s * 0.3)),
                fontsize=9, color=colors[1],
                arrowprops=dict(arrowstyle="->", color=colors[1]))

    ax.set_xticks([0, 1])
    ax.set_xticklabels(["Baseline", "Follow-up"])
    ax.set_title(f"{name} — BL → FU Trajectory", fontweight="bold")
    ax.set_ylabel(units)
    ax.legend()

plt.suptitle("Follow-up Clinical Comparison — Active vs Non-DiaLA",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("comparison_module4_followup_all.png", dpi=150, bbox_inches="tight")
plt.close()

In [ ]:
# MODULE 5: SES AS CONFOUNDER — ACTIVE vs NON-DIALA (ALL OUTCOMES)

print("=" * 70)
print("MODULE 5: SES COMPARISON & SES vs DELTA — ACTIVE vs NON-DIALA")
print("=" * 70)

active   = df_combined[df_combined["Group"] == "Active"]
nondiala = df_combined[df_combined["Group"] == "Non-DiaLA"]
ses_order    = ["Low (1–4)", "Middle (5–7)", "High (8–10)"]
bar_colors   = ["#d9534f", "#f0ad4e", "#5cb85c"]
group_colors = ["#5cb85c", "#2E86AB"]   # Active=green, Non-DiaLA=blue

delta_outcomes = {
    "HbA1c":             ("Delta_HbA1c",            "negative"),
    "BMI":               ("Delta_BMI",               "negative"),
    "HDL":               ("Delta_HDL",               "positive"),
    "TGL":               ("Delta_TGL",               "negative"),
    "Serum_Cholesterol": ("Delta_Serum_Cholesterol",  "negative"),
}

# 5a. Does SES differ between Active and Non-DiaLA?

print("\n── 5a. SES Distribution — Active vs Non-DiaLA ───────────────────")

a_ses  = active["Kupp_Occupation"].dropna()
nd_ses = nondiala["Kupp_Occupation"].dropna()

print(f"\n  Active     n={len(a_ses):>6}  mean={a_ses.mean():.2f}  "
      f"sd={a_ses.std():.2f}  median={a_ses.median():.2f}")
print(f"  Non-DiaLA  n={len(nd_ses):>6}  mean={nd_ses.mean():.2f}  "
      f"sd={nd_ses.std():.2f}  median={nd_ses.median():.2f}")

u_stat, u_p = stats.mannwhitneyu(a_ses, nd_ses, alternative="two-sided")
u_sig  = "***" if u_p < 0.001 else "**" if u_p < 0.01 else "*" if u_p < 0.05 else "ns"
rbc    = 1 - (2 * u_stat) / (len(a_ses) * len(nd_ses))
print(f"\n  Mann-Whitney U={u_stat:.1f}  p={u_p:.4f}  {u_sig}")
print(f"  Effect size (rank biserial): r={rbc:.3f}")
if u_p < 0.05:
    print("  ► SES differs significantly between groups — SES is a confounder")
    print("    Must account for SES when comparing outcomes")
else:
    print("  ► SES comparable between groups — less likely to confound outcomes")

# 5b. SES group breakdown + chi-square

print("\n── 5b. SES Group Breakdown ──────────────────────────────────────")
print(f"\n  {'SES Group':<20} {'Active n (%)':>16} {'Non-DiaLA n (%)':>16}")
print(f"  {'-'*54}")
for grp in ses_order:
    a_n    = (active["SES_Group_str"]   == grp).sum()
    nd_n   = (nondiala["SES_Group_str"] == grp).sum()
    a_pct  = a_n  / len(active)   * 100
    nd_pct = nd_n / len(nondiala) * 100
    print(f"  {grp:<20} {a_n:>5} ({a_pct:.1f}%)  {nd_n:>5} ({nd_pct:.1f}%)")

ct = pd.crosstab(df_combined["Group"], df_combined["SES_Group_str"])
ct = ct.reindex(columns=ses_order, fill_value=0)
chi2, chi_p, dof, _ = stats.chi2_contingency(ct)
chi_sig = "***" if chi_p < 0.001 else "**" if chi_p < 0.01 else "*" if chi_p < 0.05 else "ns"
print(f"\n  Chi-square on SES distribution: χ²={chi2:.3f}  df={dof}  "
      f"p={chi_p:.4f}  {chi_sig}")

# 5c. Spearman: SES vs Delta for each outcome within each group

print("\n── 5c. Spearman: SES vs Delta within each group ─────────────────")

for name, (delta_col, direction) in delta_outcomes.items():
    print(f"\n  {name}:")
    for grp_name, grp_df in [("Active", active), ("Non-DiaLA", nondiala)]:
        sub = grp_df[["Kupp_Occupation", delta_col]].dropna()
        if len(sub) < 3:
            print(f"    {grp_name}: insufficient data")
            continue
        r, p = stats.spearmanr(sub["Kupp_Occupation"], sub[delta_col])
        sig  = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
        print(f"    {grp_name:<12} r={r:.3f}  p={p:.4f}  {sig}  (n={len(sub)})")
        if p < 0.05:
            if direction == "negative":
                interp = "higher SES → greater improvement" if r < 0 else "higher SES → less improvement"
            else:
                interp = "higher SES → greater improvement" if r > 0 else "higher SES → less improvement"
            print(f"      ► {interp}")
        else:
            print(f"      ► No significant SES relationship")

# 5d. Delta by SES group — all outcomes

print("\n── 5d. Delta by SES Group — Active vs Non-DiaLA ─────────────────")

for name, (delta_col, direction) in delta_outcomes.items():
    print(f"\n  {name}:")
    print(f"    {'SES Group':<20} {'Active Delta':>18} {'Non-DiaLA Delta':>18}")
    print(f"    {'-'*58}")
    for grp in ses_order:
        a_d  = active[active["SES_Group_str"]   == grp][delta_col].dropna()
        nd_d = nondiala[nondiala["SES_Group_str"] == grp][delta_col].dropna()
        a_str  = f"{a_d.mean():.3f} ± {a_d.std():.3f}"  if len(a_d)  > 0 else "N/A"
        nd_str = f"{nd_d.mean():.3f} ± {nd_d.std():.3f}" if len(nd_d) > 0 else "N/A"
        print(f"    {grp:<20} {a_str:>18} {nd_str:>18}")


# Plots

n_outcomes = len(delta_outcomes)

# Plot A: SES score boxplot + SES group stacked bar (1 row, 2 panels)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
bp = ax.boxplot([a_ses.values, nd_ses.values],
                tick_labels=["Active", "Non-DiaLA"], patch_artist=True)
for patch, color in zip(bp["boxes"], group_colors):
    patch.set_facecolor(color); patch.set_alpha(0.7)
y_max = max(a_ses.max(), nd_ses.max()) + 0.5
ax.plot([1, 2], [y_max, y_max], color="black", linewidth=1)
ax.text(1.5, y_max + 0.1, u_sig, ha="center", fontsize=12, fontweight="bold")
ax.set_title("SES Score — Active vs Non-DiaLA", fontweight="bold")
ax.set_ylabel("Kuppuswamy Score")

ax = axes[1]
ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
bottom = np.zeros(2)
for grp, color in zip(ses_order, bar_colors):
    vals = ct_pct[grp].values if grp in ct_pct.columns else np.zeros(2)
    ax.bar(["Active", "Non-DiaLA"], vals, bottom=bottom,
           label=grp, color=color, alpha=0.8)
    bottom += vals
ax.set_title("SES Group Distribution %", fontweight="bold")
ax.set_ylabel("Percentage (%)")
ax.legend(title="SES Group", loc="upper right")

plt.suptitle("SES Distribution — Active vs Non-DiaLA",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("comparison_module5_ses_distribution.png", dpi=150, bbox_inches="tight")
plt.close()
print("\n✔ Plot A saved — comparison_module5_ses_distribution.png")

# Plot B: Delta by SES group per outcome — 2 panels per outcome (Active \ Non-DiaLA)
fig, axes = plt.subplots(n_outcomes, 2, figsize=(12, 5 * n_outcomes))

for row, (name, (delta_col, direction)) in enumerate(delta_outcomes.items()):
    for col_idx, (grp_name, grp_df, color) in enumerate([
        ("Active",    active,   group_colors[0]),
        ("Non-DiaLA", nondiala, group_colors[1])
    ]):
        ax   = axes[row, col_idx]
        data = [grp_df[grp_df["SES_Group_str"] == g][delta_col].dropna().values
                for g in ses_order]
        bp   = ax.boxplot(data, tick_labels=ses_order, patch_artist=True)
        for patch, bc in zip(bp["boxes"], bar_colors):
            patch.set_facecolor(bc); patch.set_alpha(0.7)
        ax.axhline(0, color="black", linestyle="--", linewidth=0.8, alpha=0.5)
        ax.set_title(f"{name} — Δ by SES Group ({grp_name})", fontweight="bold")
        ax.set_ylabel(f"Δ {name} (FU - BL)")
        ax.set_xlabel("SES Group")
        ax.tick_params(axis="x", rotation=15)

plt.suptitle("Delta by SES Group — Active vs Non-DiaLA (All Outcomes)",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("comparison_module5_delta_by_ses_all.png", dpi=150, bbox_inches="tight")
plt.close()

In [ ]:
# MODULE 6: INTERACTION REGRESSION — DOES diala MODIFY THE SES EFFECT?
# Linear model: Delta ~ Group + Kupp_Occupation + Group×Kupp_Occupation + Age_Group + Gender_Code
# Key question: Is the SES-outcome relationship different for Active vs Non-DiaLA? (interaction term = effect modification)

import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

print("=" * 72)
print("MODULE 6: INTERACTION REGRESSION — SES × GROUP EFFECT MODIFICATION")
print("  Model: Delta ~ Group + Kupp_Occupation + Group:Kupp_Occupation")
print("                + Age_Group + Gender_Code")
print("  Reference group: Non-DiaLA")
print("  Key term: Group[Active]:Kupp_Occupation")
print("    → significant = DiaLA modifies the SES-outcome relationship")
print("=" * 72)

active   = df_combined[df_combined["Group"] == "Active"].copy()
nondiala = df_combined[df_combined["Group"] == "Non-DiaLA"].copy()

# Group must be coded for regression: Non-DiaLA = 0 (ref), Active = 1
df_reg = df_combined.copy()
df_reg["Group_bin"] = (df_reg["Group"] == "Active").astype(int)

delta_outcomes = {
    "HbA1c":             ("Delta_HbA1c",            "negative", "HbA1c (%)"),
    "BMI":               ("Delta_BMI",               "negative", "BMI (kg/m²)"),
    "HDL":               ("Delta_HDL",               "positive", "HDL (mg/dL)"),
    "TGL":               ("Delta_TGL",               "negative", "TGL (mg/dL)"),
    "Serum_Cholesterol": ("Delta_Serum_Cholesterol",  "negative", "Serum Chol (mg/dL)"),
}

module6_results = []

for name, (delta_col, direction, units) in delta_outcomes.items():

    sub = df_reg[[delta_col, "Group_bin", "Kupp_Occupation",
                  "Age_Group", "Gender_Code"]].dropna().copy()
    n   = len(sub)

    if n < 20:
        print(f"\n{name}: insufficient data (n={n}), skipping.")
        continue

    print(f"\n{'─' * 72}")
    print(f"{name}  |  n={n:,}  |  Outcome unit: {units}")

    # Fit model with interaction 
    formula = (f"{delta_col} ~ Group_bin * Kupp_Occupation "
               f"+ Age_Group + Gender_Code")
    model   = smf.ols(formula, data=sub).fit()

    #  Extract key terms 
    terms = {
        "Group (Active vs Non-DiaLA)":    "Group_bin",
        "SES (Kupp score)":               "Kupp_Occupation",
        "SES × Group interaction":        "Group_bin:Kupp_Occupation",
        "Age group":                      "Age_Group",
        "Gender":                         "Gender_Code",
    }

    print(f"\n  {'Term':<35} {'Coef':>8}  {'95% CI':>18}  {'p':>8}  {'Sig':>4}")
    print(f"  {'-'*35} {'-'*8}  {'-'*18}  {'-'*8}  {'-'*4}")

    ci = model.conf_int()

    for label, term in terms.items():
        if term not in model.params:
            continue
        coef = model.params[term]
        lo   = ci.loc[term, 0]
        hi   = ci.loc[term, 1]
        p    = model.pvalues[term]
        sig  = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
        print(f"  {label:<35} {coef:>8.3f}  ({lo:.3f}, {hi:.3f})  {p:>8.4f}  {sig}")

    #  R² and model fit 
    print(f"\n  R² = {model.rsquared:.4f}  |  Adj R² = {model.rsquared_adj:.4f}"
          f"  |  F-stat p = {model.f_pvalue:.4f}")

    #  Interpret interaction term 
    interaction_coef = model.params.get("Group_bin:Kupp_Occupation", np.nan)
    interaction_p    = model.pvalues.get("Group_bin:Kupp_Occupation", np.nan)
    interaction_sig  = ("***" if interaction_p < 0.001 else
                        "**"  if interaction_p < 0.01  else
                        "*"   if interaction_p < 0.05  else "ns")

    print(f"\n  ── INTERACTION INTERPRETATION ──────────────────────────────")
    if interaction_p < 0.05:
        if direction == "negative":
            # More negative delta = better; negative interaction coef means
            # Active has steeper SES gradient (higher SES → bigger drop)
            if interaction_coef < 0:
                interp = ("Active users show a STEEPER SES gradient — higher SES "
                          "predicts greater improvement in Active but not Non-DiaLA. "
                          "DiaLA may amplify SES advantage.")
            else:
                interp = ("Active users show a FLATTER SES gradient — DiaLA attenuates "
                          "the SES effect, equalling outcomes across SES groups.")
        else:  # positive direction (HDL)
            if interaction_coef > 0:
                interp = ("Active users show a STEEPER SES gradient — higher SES "
                          "predicts greater HDL improvement in Active users.")
            else:
                interp = ("Active users show a FLATTER SES gradient — DiaLA attenuates "
                          "the SES effect on HDL across groups.")
        print(f"  ► SIGNIFICANT interaction (p={interaction_p:.4f} {interaction_sig})")
        print(f"  ► {interp}")
    else:
        print(f"  ► No significant interaction (p={interaction_p:.4f} {interaction_sig})")
        print(f"  ► SES effect on {name} does not differ between Active and Non-DiaLA")
        print(f"  ► DiaLA does not appear to modify the SES-{name} relationship")

    module6_results.append({
        "Variable":          name,
        "n":                 n,
        "Interaction_coef":  round(interaction_coef, 4) if not np.isnan(interaction_coef) else np.nan,
        "Interaction_p":     round(interaction_p, 4)    if not np.isnan(interaction_p)    else np.nan,
        "Interaction_sig":   interaction_sig,
        "R2":                round(model.rsquared, 4),
        "direction":         direction,
        "units":             units,
        "model":             model,
        "delta_col":         delta_col,
    })

#  Summary table 
print(f"\n{'=' * 72}")
print("MODULE 6 SUMMARY — INTERACTION TERMS (SES × Group)")
print(f"{'=' * 72}")
print(f"  {'Variable':<22} {'Interaction Coef':>18} {'p':>8} {'Sig':>5} "
      f"{'R²':>6}  {'Interpretation'}")
print(f"  {'-'*22} {'-'*18} {'-'*8} {'-'*5} {'-'*6}  {'-'*30}")
for r in module6_results:
    sig = r["Interaction_sig"]
    interp = "DiaLA modifies SES effect" if sig != "ns" else "No modification"
    print(f"  {r['Variable']:<22} {r['Interaction_coef']:>18} "
          f"{r['Interaction_p']:>8} {sig:>5} {r['R2']:>6}  {interp}")

print(f"\n  Interaction coef > 0: Active has stronger positive SES gradient")
print(f"  Interaction coef < 0: Active has stronger negative SES gradient")
print(f"  Key: sign depends on outcome direction (negative = improvement for most)")

# PLOTS: Predicted delta by SES score for each group (interaction plot)

n_plots = len(module6_results)
fig, axes = plt.subplots(1, n_plots, figsize=(6 * n_plots, 5), sharey=False)
if n_plots == 1:
    axes = [axes]

colors = ["#5cb85c", "#2E86AB"]
ses_range = np.linspace(1, 10, 100)

# Fix Age_Group and Gender_Code at median for prediction
median_age    = df_reg["Age_Group"].median()
median_gender = df_reg["Gender_Code"].median()

for ax, r in zip(axes, module6_results):
    model     = r["model"]
    name      = r["Variable"]
    units     = r["units"]
    sig       = r["Interaction_sig"]

    for group_val, label, color in [(0, "Non-DiaLA", colors[1]),
                                     (1, "Active",    colors[0])]:
        pred_df = pd.DataFrame({
            "Group_bin":       group_val,
            "Kupp_Occupation": ses_range,
            "Age_Group":       median_age,
            "Gender_Code":     median_gender,
        })
        pred  = model.predict(pred_df)
        ax.plot(ses_range, pred, color=color, linewidth=2.5, label=label)

    ax.axhline(0, color="grey", linestyle="--", linewidth=0.8, alpha=0.6)
    ax.set_xlabel("Kuppuswamy SES Score", fontsize=10)
    ax.set_ylabel(f"Predicted Δ {units}", fontsize=10)
    ax.set_title(f"{name}\nInteraction: {sig}", fontweight="bold", fontsize=11)
    ax.legend(fontsize=9)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

plt.suptitle(
    "SES × Group Interaction — Predicted Clinical Improvement\n"
    "Active vs Non-DiaLA  (controls: median age & gender)",
    fontsize=13, fontweight="bold", y=1.02
)
plt.tight_layout()
plt.savefig("comparison_module6_interaction_plots.png", dpi=150, bbox_inches="tight")
plt.close()